# 02 - Feature Engineering
## IEEE-CIS Fraud Detection

Transform raw Kaggle data into model-ready features:
- Categorical encoding (label + frequency)
- Time-based features
- Transaction amount transformations
- Aggregation features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, '..')

from src.config import (
    TRAIN_TRANSACTION_FILE,
    TRAIN_IDENTITY_FILE,
    TARGET_COL,
    ID_COL,
)
from src.features.build_features import FeatureEngineer

print("✅ Setup complete!")

✅ Setup complete!


## 1. Load Kaggle Data

In [ ]:
# Load and merge data
print("📂 Loading Kaggle data...")
train_txn = pd.read_csv(TRAIN_TRANSACTION_FILE)
train_id = pd.read_csv(TRAIN_IDENTITY_FILE)

df = train_txn.merge(train_id, on='TransactionID', how='left')
print(f"   Shape: {df.shape}")
print(f"   Fraud rate: {df[TARGET_COL].mean():.2%}")

📂 Loading Kaggle data...
   Shape: (590540, 434)
   Fraud rate: 3.50%


## 2. Apply Feature Engineering Pipeline

In [ ]:
# Apply feature engineering
fe = FeatureEngineer()
df_features = fe.fit_transform(df)

print(f"📊 Feature Engineering Results:")
print(f"   Original columns: {len(df.columns)}")
print(f"   After engineering: {len(df_features.columns)}")
print(f"   New features added: {len(df_features.columns) - len(df.columns)}")

2026-01-20 20:30:05,815 - INFO - Fitting feature engineer...
2026-01-20 20:30:06,779 - INFO -   Fitted 22 categorical encoders


📊 Feature Engineering Results:
   Original columns: 434
   After engineering: 487
   New features added: 53


In [ ]:
# Show new engineered features
new_cols = [c for c in df_features.columns if c not in df.columns]
print(f"🆕 New Engineered Features ({len(new_cols)}):")
for col in new_cols[:20]:
    print(f"   • {col}")
if len(new_cols) > 20:
    print(f"   ... and {len(new_cols) - 20} more")

🆕 New Engineered Features (53):
   • ProductCD_encoded
   • ProductCD_freq
   • card1_encoded
   • card1_freq
   • card2_encoded
   • card2_freq
   • card3_encoded
   • card3_freq
   • card4_encoded
   • card4_freq
   • card5_encoded
   • card5_freq
   • card6_encoded
   • card6_freq
   • addr1_encoded
   • addr1_freq
   • addr2_encoded
   • addr2_freq
   • P_emaildomain_encoded
   • P_emaildomain_freq
   ... and 33 more


## 3. Feature Transformations Explained

In [ ]:
# Show example transformations
print("🔄 Feature Transformation Examples:\n")

# Amount features
print("1. TRANSACTION AMOUNT FEATURES")
sample = df_features[['TransactionAmt', 'TransactionAmt_log', 'TransactionAmt_decimal', 'TransactionAmt_is_round']].head()
print(sample.to_string())

# Time features
print("\n2. TIME FEATURES (from TransactionDT)")
if 'hour' in df_features.columns:
    sample = df_features[['TransactionDT', 'hour', 'day', 'is_weekend', 'is_night']].head()
    print(sample.to_string())

🔄 Feature Transformation Examples:

1. TRANSACTION AMOUNT FEATURES
   TransactionAmt  TransactionAmt_log  TransactionAmt_decimal  TransactionAmt_is_round
0            68.5            4.241327                     0.5                        0
1            29.0            3.401197                     0.0                        1
2            59.0            4.094345                     0.0                        1
3            50.0            3.931826                     0.0                        1
4            50.0            3.931826                     0.0                        1

2. TIME FEATURES (from TransactionDT)
   TransactionDT  hour  day  is_weekend  is_night
0          86400     0    1           0         1
1          86401     0    1           0         1
2          86469     0    1           0         1
3          86499     0    1           0         1
4          86506     0    1           0         1


## 4. Encoding Strategies

In [ ]:
# Show encoding examples
print("🏷️ Categorical Encoding Examples:\n")

# Label encoding
print("Label Encoding (P_emaildomain):")
if 'P_emaildomain_encoded' in df_features.columns:
    sample = df_features[['P_emaildomain', 'P_emaildomain_encoded', 'P_emaildomain_freq']].dropna().drop_duplicates().head(10)
    print(sample.to_string())

# Frequency encoding explanation
print("\n📊 Frequency Encoding captures 'rarity' of values:")
print("   gmail.com (common) → high frequency score")
print("   rare-domain.com → low frequency score")
print("   This helps model identify unusual patterns!")

🏷️ Categorical Encoding Examples:

Label Encoding (P_emaildomain):
    P_emaildomain  P_emaildomain_encoded  P_emaildomain_freq
0         MISSING                      0            0.000000
1       gmail.com                     17            0.460315
2     outlook.com                     36            0.010272
3       yahoo.com                     54            0.203462
7        mail.com                     30            0.001127
8   anonymous.com                      2            0.074580
11    hotmail.com                     20            0.091214
12    verizon.net                     49            0.005453
13        aol.com                      3            0.057025
26         me.com                     31            0.003068

📊 Frequency Encoding captures 'rarity' of values:
   gmail.com (common) → high frequency score
   rare-domain.com → low frequency score
   This helps model identify unusual patterns!


## 5. Feature Statistics

In [ ]:
# Feature statistics
numeric_features = df_features.select_dtypes(include=[np.number]).columns
feature_cols = [c for c in numeric_features if c not in [ID_COL, TARGET_COL]]

print(f"📈 Final Feature Set:")
print(f"   Total numeric features: {len(feature_cols)}")
print(f"   Ready for model training!")

# Show correlation of new features with target
new_numeric = [c for c in new_cols if c in numeric_features]
if new_numeric:
    correlations = df_features[new_numeric].corrwith(df_features[TARGET_COL]).abs().sort_values(ascending=False)
    print(f"\n🎯 New Features Correlation with Fraud (top 10):")
    for feat, corr in correlations.head(10).items():
        print(f"   {feat}: {corr:.4f}")

📈 Final Feature Set:
   Total numeric features: 452
   Ready for model training!

🎯 New Features Correlation with Fraud (top 10):
   addr2_freq: 0.1634
   R_emaildomain_freq: 0.1618
   ProductCD_encoded: 0.1566
   card3_freq: 0.1522
   card3_encoded: 0.1443
   DeviceType_encoded: 0.1438
   ProductCD_freq: 0.1303
   DeviceType_freq: 0.1222
   M6_freq: 0.1195
   addr2_encoded: 0.1153


## 6. Save Processed Data (Optional)

In [ ]:
# Optionally save processed features
# Uncomment to save:

# from src.config import PROCESSED_DATA_DIR
# PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
# df_features.to_parquet(PROCESSED_DATA_DIR / 'train_features.parquet', index=False)
# print(f"✅ Saved to {PROCESSED_DATA_DIR / 'train_features.parquet'}")

print("💡 To process data via CLI, run: make make_dataset")

💡 To process data via CLI, run: make make_dataset
